[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch13-model-serving/03-phuc-vu-llm.ipynb)

# Phục vụ LLM: hai pha, bộ nhớ đệm KV và gom batch liên tục

Chương 13 nói mô hình ngôn ngữ lớn làm ba tính chất nổi bật hẳn lên: sinh tự hồi quy (mỗi token phụ
thuộc mọi token trước), đầu ra có độ dài không biết trước, và một bộ nhớ có trạng thái, bộ nhớ đệm
KV (KV cache), lớn lên theo mỗi token. Lý thuyết hàng đợi và thuế gom batch của hai sổ tay trước vẫn
đúng, nhưng chúng áp vào hai pha khác nhau của một yêu cầu: giai đoạn xử lý prompt (prefill) và giai
đoạn sinh token (decode).

**Bạn sẽ làm:**
1. đổi các đích độ trễ của chương ra con số, rồi tính lại ca Llama 3 8B: TTFT, mức sàn của mỗi bước
   decode theo phương trình 18, và so sánh với A100;
2. đo trên CPU của bạn một bước decode ở nhiều kích thước batch, và tìm chỗ nó đổi từ nghẽn ở băng
   thông sang nghẽn ở tính toán;
3. tính bộ nhớ đệm KV mỗi token, mỗi yêu cầu và sức chứa của GPU, rồi vẽ lại hình 7;
4. dựng một mô hình của decode theo batch và xem batch bị chặn bởi bộ nhớ hay bởi đích TPOT;
5. mô phỏng gom batch tĩnh và gom batch liên tục, rồi cấp bộ nhớ đệm KV liền khối và theo khối;
6. tính lợi ích của lưu đệm tiền tố, năng lượng của một cuộc trò chuyện, và giá thành mỗi token.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
import heapq

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]  # thứ tự màu cố định
GB, GiB = 1e9, 2 ** 30

# Cấu hình công khai của hai mô hình trong chương (giả định của sổ tay: chương chỉ đưa kết quả).
# tang: số tầng transformer, dau_kv: số đầu key/value, dau_q: số đầu truy vấn, chieu_dau: số chiều mỗi đầu.
LLAMA3_8B = dict(tang=32, dau_kv=8, dau_q=32, chieu_dau=128)
LLAMA_70B = dict(tang=80, dau_kv=8, dau_q=64, chieu_dau=128)


def kv_moi_token(cfg, byte=2, dau=None):
    """Byte bộ nhớ đệm KV cho một token: K và V, ở mọi tầng, mọi đầu KV. byte=2 là FP16."""
    return 2 * cfg["tang"] * (dau or cfg["dau_kv"]) * cfg["chieu_dau"] * byte


def thoi_gian(f, lap=7, khoi_dong=2):
    """Trung vị thời gian của f() qua nhiều lần chạy, tính bằng giây."""
    for _ in range(khoi_dong):
        f()
    ts = []
    for _ in range(lap):
        t1 = time.perf_counter()
        f()
        ts.append(time.perf_counter() - t1)
    return float(np.median(ts))


print(f"KV mỗi token: Llama 3 8B {kv_moi_token(LLAMA3_8B):,} byte, mô hình 70 tỷ {kv_moi_token(LLAMA_70B):,} byte (FP16)")

## 1. Hai pha, hai đồng hồ

Giai đoạn xử lý prompt chạy song song trên mọi token của prompt và điền bộ nhớ đệm KV cho chúng; nó
quyết định **thời gian tới token đầu tiên** (TTFT), lúc chữ đầu tiên hiện ra. Giai đoạn sinh token
sinh từng token một; nó quyết định **thời gian mỗi token đầu ra** (TPOT), nhịp của dòng chữ. Góc nhìn
hệ thống 1.8 cho ba đích minh hoạ, và checkpoint 1.4 hỏi đổi đích TPOT ra token mỗi giây.

In [ ]:
TTFT_DICH = sach(500, nguon="1.8 · 500 ms")
PROMPT_DICH = sach(1000, nguon="500 ms · 1,000 token")
TPOT_DICH = sach(50, nguon="1,000 token · 50 ms")
TONG_DICH = sach(1000, trich="Throughput: \\(>\\) 1000 tokens/s aggregate across active serving replicas")
theo_sach("TPOT 50 ms đổi ra token mỗi giây", 1 / sach(0.05, nguon="50 ms · 0.05 giây"), sach=20, nguon="0.05 giây · 20 token")
print(f"đích: TTFT < {TTFT_DICH} ms cho prompt {PROMPT_DICH:,} token · TPOT < {TPOT_DICH} ms · tổng > {TONG_DICH:,} token/s")
print(f"một câu trả lời 256 token ở đúng đích TPOT kéo dài {256 * TPOT_DICH / 1000:.1f} s sau token đầu tiên")
print(f"cần ít nhất {math.ceil(TONG_DICH / (1000 / TPOT_DICH))} luồng sinh song song để đạt thông lượng tổng ở nhịp ấy")

## 2. Llama 3 8B trên một H100

Ca nghiên cứu của chương: Llama 3 với 8 tỷ tham số, lượng tử hóa 4 bit bằng AWQ (chỉ trọng số), trên
một GPU H100 SXM5 có 80 GB HBM3 và băng thông 3.35 TB/s. Mỗi yêu cầu có prompt 1,000 token và câu trả
lời 256 token; đích là TTFT dưới 200 ms và TPOT dưới 20 ms.

Với decode ở batch 1, mỗi bước đọc toàn bộ trọng số một lần để sinh một token. Mật độ tính toán vì thế
là 2 FLOP mỗi tham số chia cho nửa byte mỗi tham số, và phương trình 18 cho mức sàn của mỗi bước:

$$T_\text{token} \approx \frac{D_\text{vol}}{\text{BW}_\text{memory}}.$$

**Dự đoán:** bỏ qua mọi thứ trừ việc đọc trọng số, H100 sinh được tối đa bao nhiêu token mỗi giây cho
một người dùng?

In [ ]:
THAM_SO = sach(8e9, nguon="4.3 nghìn · 8 tỷ")
BIT = sach(4, nguon="8 tỷ · 4 bit")
W_BYTE = sach(4, nguon="18 · 4 GB") * GB
BW_H100 = sach(3.35, nguon="4 GB · 3.35 TB/s") * 1e12
PROMPT = sach(1000, nguon="3.35 TB/s · 1,000 token")
TRA_LOI = sach(256, nguon="1,000 token · 256 token")
SLO_TTFT = sach(200, nguon="256 token · 200 ms")
SLO_TPOT = sach(20, nguon="200 ms · 20 ms")
assert THAM_SO * BIT / 8 == W_BYTE                           # 8 tỷ × nửa byte = 4 GB

FLOP_THAM_SO = sach(2, trich="approximately \\(2\\) FLOPs per parameter divided by \\(0.5\\) byte per parameter, or \\(4\\) FLOPs/byte")
BYTE_THAM_SO = sach(0.5, trich="approximately \\(2\\) FLOPs per parameter divided by \\(0.5\\) byte per parameter, or \\(4\\) FLOPs/byte")
theo_sach("mật độ tính toán của decode ở batch 1 (FLOP/byte)", FLOP_THAM_SO / BYTE_THAM_SO, sach=4,
          trich="approximately \\(2\\) FLOPs per parameter divided by \\(0.5\\) byte per parameter, or \\(4\\) FLOPs/byte")
T_SAN = W_BYTE / BW_H100
theo_sach("mức sàn mỗi token trên H100 (ms)", 1000 * T_SAN, sach=1.2, nguon="3.35 TB/s · 1.2 ms")
TPOT_MH = sach(1.57, nguon="1.2 ms · 1.57 ms") / 1000      # cộng thuế điều phối theo tầng của bộ giải
theo_sach("sinh 256 token (giây)", TRA_LOI * TPOT_MH, sach=0.40, nguon="256 · 0.40 giây")
print(f"trần cho một người dùng: {1 / T_SAN:,.0f} token/s theo mức sàn, {1 / TPOT_MH:,.0f} token/s với TPOT 1.57 ms")
print(f"thuế điều phối suy ra: {1000 * (TPOT_MH - T_SAN):.2f} ms mỗi bước, {100 * (TPOT_MH - T_SAN) / TPOT_MH:.0f} % của TPOT")

TTFT_MH = sach(41.9, nguon="40 % · 41.9 ms") / 1000
HIEU_SUAT = sach(40, nguon="1,000 token · 40 %") / 100
F_HD = FLOP_THAM_SO * THAM_SO * PROMPT / TTFT_MH            # FLOP/s hiệu dụng mà 41.9 ms ngầm định
print(f"TTFT {1000 * TTFT_MH} ms < SLO {SLO_TTFT} ms; TPOT {1000 * TPOT_MH} ms < SLO {SLO_TPOT} ms")
print(f"suy ngược: prefill 1,000 token là {FLOP_THAM_SO * THAM_SO * PROMPT / 1e12:.0f} TFLOP, nên 41.9 ms ngầm định"
      f" {F_HD / 1e12:.0f} TFLOP/s hiệu dụng, tức đỉnh {F_HD / HIEU_SUAT / 1e12:.0f} TFLOP/s ở hiệu suất 40 %")
print(f"phía mô hình, một yêu cầu trọn vẹn: {1000 * TTFT_MH:.1f} + {1000 * TRA_LOI * TPOT_MH:.1f} = {1000 * (TTFT_MH + TRA_LOI * TPOT_MH):.1f} ms")

Chương không cho đỉnh FLOP/s của H100, nên 41.9 ms không tính lại được từ các số trong chương. Ô trên
đi theo chiều ngược lại: từ 41.9 ms và hiệu suất 40 %, nó suy ra tốc độ tính mà bộ giải của sách đã
dùng, khoảng 2 FLOP cho mỗi tham số và mỗi token. Phần sau dùng tốc độ hiệu dụng ấy như một giả định.

Mức sàn 1.2 ms cho thấy điều chương nhấn mạnh: decode nghẽn ở băng thông, nên thêm lõi tính toán gần
như không giảm độ trễ. Thứ đổi được giới hạn là bộ nhớ nhanh hơn, mô hình nhỏ hơn, hoặc gom batch.
Trên A100 80 GB với 2.04 TB/s:

In [ ]:
BW_A100 = sach(2.04, nguon="80 GB · 2.04 TB/s") * 1e12
SAN_A100 = sach(1.97, nguon="2.04 TB/s · 1.97 ms")
print(f"4 GB / 2.04 TB/s = {1000 * W_BYTE / BW_A100:.3f} ms; sách ghi {SAN_A100} ms")
THAM_SO_CONG_KHAI = 8.03e9           # số tham số công khai của Llama 3 8B, giả định của sổ tay
print(f"với {THAM_SO_CONG_KHAI / 1e9} tỷ tham số × 0.5 byte = {THAM_SO_CONG_KHAI * 0.5 / GB:.3f} GB:"
      f" A100 {1000 * THAM_SO_CONG_KHAI * 0.5 / BW_A100:.3f} ms, H100 {1000 * THAM_SO_CONG_KHAI * 0.5 / BW_H100:.3f} ms")
print(f"A100 chậm hơn H100 {BW_H100 / BW_A100:.2f} lần ở mỗi bước, đúng bằng tỉ lệ băng thông")

Với đúng 4 GB, phép chia cho 1.96 ms, không phải 1.97. Hai con số chỉ khớp nếu trọng số nặng hơn
4 GB một chút: với số tham số công khai của Llama 3 8B, phép chia cho 1.97 ms trên A100 mà vẫn cho 1.2
ms trên H100. Có lẽ bộ giải của sách dùng số tham số chính xác, còn văn bản làm tròn thành 4 GB; ô trên
in cả hai thay vì đối chiếu.

## 3. Một bước decode trên CPU của bạn

Trên CPU không có HBM, nhưng có cùng hai giới hạn: băng thông bộ nhớ và tốc độ tính. Ô dưới dựng một
chồng 8 ma trận trọng số 2048 × 2048 ở FP32 (giả định của sổ tay, khoảng 134 MB, lớn hơn bộ nhớ đệm
của CPU), rồi đo một bước "decode" đi qua cả chồng ở nhiều kích thước batch. Mỗi bước đọc toàn bộ
trọng số một lần, dù batch có bao nhiêu chuỗi.

**Dự đoán:** khi batch tăng từ 1 lên 128, thời gian một bước tăng gần 128 lần, hay ít hơn nhiều?

In [ ]:
torch.set_num_threads(1)
torch.manual_seed(0)
D_CPU, TANG_CPU = 2048, 8
trong_so = [torch.randn(D_CPU, D_CPU) / D_CPU ** 0.5 for _ in range(TANG_CPU)]
BYTE_CPU = sum(w.numel() * w.element_size() for w in trong_so)
THAM_SO_CPU = sum(w.numel() for w in trong_so)


def buoc_cpu(x):
    for w in trong_so:
        x = torch.nn.functional.linear(x, w)
    return x


CAC_B_CPU = np.array([1, 2, 4, 8, 16, 32, 64, 128])
tg_cpu = []
with torch.inference_mode():
    for b in CAC_B_CPU:
        xb = torch.randn(int(b), D_CPU)
        tg_cpu.append(thoi_gian(lambda: buoc_cpu(xb)))
    x_prompt = torch.randn(256, D_CPU)
    tg_prompt = thoi_gian(lambda: buoc_cpu(x_prompt), lap=5)
tg_cpu = np.array(tg_cpu)
bang_thong = BYTE_CPU / tg_cpu                        # byte trọng số đọc mỗi giây
tinh = 2 * THAM_SO_CPU * CAC_B_CPU / tg_cpu          # FLOP mỗi giây
BW_CPU, F_CPU = bang_thong.max(), tinh.max()          # hai "mái": mức tốt nhất đạt được
do_tren_may("trọng số của chồng", BYTE_CPU / 1e6, "MB")
for b, t in zip(CAC_B_CPU, tg_cpu):
    do_tren_may(f"một bước ở batch {b:>3}", 1000 * t, "ms")
do_tren_may("băng thông tốt nhất (mái băng thông)", BW_CPU / 1e9, "GB/s")
do_tren_may("tốc độ tính tốt nhất (mái tính toán)", F_CPU / 1e9, "GFLOP/s")
do_tren_may("điểm gãy: mật độ tính toán nơi hai mái gặp nhau", F_CPU / BW_CPU, "FLOP/byte")
do_tren_may("batch ở điểm gãy (FP32: mật độ = batch / 2)", 2 * F_CPU / BW_CPU, "")
do_tren_may("prefill 256 token một lần so với 256 bước decode ở batch 1", 256 * tg_cpu[0] / tg_prompt, "lần nhanh hơn")

fig, ax = plt.subplots(figsize=(7.5, 4.4))
b_ve = np.logspace(0, np.log10(256), 100)
ax.loglog(b_ve, b_ve * BW_CPU / BYTE_CPU, color=MAU[1], lw=1.5, ls="--", label="mái băng thông: batch × BW / byte trọng số")
ax.loglog(b_ve, np.full_like(b_ve, F_CPU / (2 * THAM_SO_CPU)), color=MAU[2], lw=1.5, ls="--",
          label="mái tính toán: FLOP/s / (2 × tham số)")
ax.loglog(CAC_B_CPU, CAC_B_CPU / tg_cpu, "o-", color=MAU[0], lw=2, ms=7, mec="white", mew=1.2, label="đo")
ax.set_xticks(CAC_B_CPU, [str(b) for b in CAC_B_CPU])
ax.set_xlabel("số chuỗi trong batch (thang log)")
ax.set_ylabel("token sinh ra mỗi giây (thang log)")
ax.set_title("Một bước decode trên CPU (đo trên máy này)")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

Mỗi mái được dựng từ mức tốt nhất mà chính phép đo đạt được, nên mọi điểm đo nằm dưới cả hai. Dưới
điểm gãy, giới hạn là mái băng thông, và nó tăng tỉ lệ với batch: cùng một lần đọc trọng số phục vụ
nhiều chuỗi hơn. Qua điểm gãy, giới hạn là mái tính toán, và nó không tăng nữa. Khoảng cách giữa điểm
đo và mái là phần hiệu suất mà thư viện tính toán để lại. Trên nhiều CPU, batch 1 còn đi một đường
riêng (nhân ma trận với vectơ) nhanh hơn hẳn những batch nhỏ ngay sau nó.

CPU có điểm gãy rất thấp vì tốc độ tính của một luồng nhỏ so với băng thông của nó. Một GPU như H100 có
tốc độ tính lớn hơn rất nhiều so với băng thông, nên điểm gãy ở xa: với mật độ 4 FLOP mỗi byte ở batch
1, decode còn nằm trên dốc băng thông qua cả một dải batch dài, và gom batch tăng thông lượng gần như
miễn phí trong dải ấy. Prefill thì khác: nó đưa cả prompt qua mô hình trong một lần, nên nó chạy như một
batch lớn và nằm ở mái tính toán. Đó là lý do chương đặt TTFT trên trần tính toán và TPOT trên dốc băng
thông.

## 4. Bộ nhớ đệm KV

Mỗi token đã xử lý để lại một cặp key và value ở mọi tầng. Chương cho kết quả: khoảng 0.131 MB mỗi
token ở FP16 cho Llama 3 8B, nhờ cơ chế tập trung theo nhóm truy vấn (GQA) dùng ít đầu KV hơn số đầu
truy vấn. Ô dưới tính con số ấy từ cấu hình công khai của mô hình (giả định của sổ tay, chương không
nêu cấu hình).

In [ ]:
KV8 = kv_moi_token(LLAMA3_8B)
theo_sach("KV mỗi token, Llama 3 8B, FP16 (MB)", KV8 / 1e6, sach=0.131, nguon="4 GB · 0.131 MB")
KV8_MHA = kv_moi_token(LLAMA3_8B, dau=LLAMA3_8B["dau_q"])
print(f"nếu mọi đầu truy vấn có KV riêng (MHA): {KV8_MHA / 1e6:.3f} MB, gấp {KV8_MHA / KV8:.0f} lần")
MOI_YEU_CAU = sach(1256, nguon="0.6 triệu · 1,256 token")
assert MOI_YEU_CAU == PROMPT + TRA_LOI
print(f"một yêu cầu {MOI_YEU_CAU:,} token: {MOI_YEU_CAU * KV8 / 1e6:.1f} MB bộ nhớ đệm KV ở cuối lượt sinh")
print(f"trọng số INT4 {W_BYTE / GB:.0f} GB tương đương bộ nhớ đệm KV của {W_BYTE / (MOI_YEU_CAU * KV8):.0f} yêu cầu như thế")

DU_TRU = sach(72, nguon="0.131 MB · 72 GB")
for ten, don_vi in (("GB (10^9 byte)", GB), ("GiB (2^30 byte)", GiB)):
    so_token = DU_TRU * don_vi / KV8
    print(f"72 {ten}: {so_token:,.0f} token, {so_token / 1e6:.2f} triệu, {math.floor(so_token / MOI_YEU_CAU)} yêu cầu")
token_GiB = DU_TRU * GiB / KV8
theo_sach("sức chứa, đọc 72 GB là GiB (triệu token)", token_GiB / 1e6, sach=0.6, nguon="72 GB · 0.6 triệu")
theo_sach("batch đồng thời, đọc 72 GB là GiB", math.floor(token_GiB / MOI_YEU_CAU), sach=469, nguon="1,256 token · 469 yêu cầu")

Hai con số "khoảng 0.6 triệu token" và "469 yêu cầu" chỉ khép lại khi đọc phần dự trữ 72 GB là GiB:
khi ấy phép chia cho đúng 589,824 token và 469 yêu cầu. Đọc là GB (đơn vị của 80 GB và 4 GB ngay bên
cạnh), phép chia cho khoảng 0.55 triệu token và 437 yêu cầu. Sách trộn hai đơn vị trong cùng một đoạn,
nên bài giảng gọi các con số này là ước lượng của sách. Chênh nhau 7 %, đủ nhỏ để không đổi kết luận:
một GPU giữ được vài trăm yêu cầu cùng lúc, và chính dung lượng bộ nhớ đệm KV quyết định con số ấy.

### Hình 7: batch và độ dài ngữ cảnh tranh nhau một ngân sách

Hình 7 vẽ ở quy mô 70 tỷ tham số: 35 GB trọng số INT4 trên GPU 80 GB, còn 45 GB cho bộ nhớ đệm KV, và
batch 32 chạm ngân sách ấy ở khoảng 4.3 nghìn token. Ô dưới vẽ lại hình từ công thức, với cấu hình
công khai của một mô hình 70 tỷ tham số (giả định của sổ tay: 80 tầng, 8 đầu KV, 128 chiều mỗi đầu).

In [ ]:
THAM_SO_70 = sach(70e9, nguon="70 tỷ · 35 GB")
W70 = sach(35, nguon="70 tỷ · 35 GB") * GB
assert THAM_SO_70 * BYTE_THAM_SO == W70
GPU_80 = sach(80, nguon="35 GB · 80 GB") * GB
NGAN_SACH_KV = sach(45, nguon="80 GB · 45 GB") * GB
assert GPU_80 - W70 == NGAN_SACH_KV
KV70 = kv_moi_token(LLAMA_70B)
B_HINH7 = sach(32, nguon="45 GB · 32")
theo_sach("batch 32 chạm 45 GB ở (nghìn token)", NGAN_SACH_KV / (B_HINH7 * KV70) / 1000, sach=4.3, nguon="32 · 4.3 nghìn")

ngu_canh = np.linspace(0, 32_000, 400)
fig, ax = plt.subplots(figsize=(8, 4.4))
for b, mau in zip((1, 8, 32, 128), MAU):
    ax.plot(ngu_canh / 1000, b * ngu_canh * KV70 / GB, color=mau, lw=2, label=f"batch {b}")
ax.axhline(NGAN_SACH_KV / GB, color="0.3", lw=1.2, ls="--")
ax.text(31.5, NGAN_SACH_KV / GB + 2, "45 GB còn lại sau trọng số", ha="right", fontsize=8, color="0.3")
x32 = NGAN_SACH_KV / (B_HINH7 * KV70)
ax.plot(x32 / 1000, NGAN_SACH_KV / GB, "o", color=MAU[2], ms=8, mec="white", mew=1.5)
ax.annotate(f"batch 32: {x32 / 1000:.1f} nghìn token", (x32 / 1000, NGAN_SACH_KV / GB), xytext=(15, 25),
            textcoords="offset points", fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.set_xlim(0, 32)
ax.set_ylim(0, 125)
ax.set_xlabel("độ dài ngữ cảnh mỗi chuỗi (nghìn token)")
ax.set_ylabel("bộ nhớ đệm KV của cả batch (GB)")
ax.set_title("Hình 7 vẽ lại từ công thức: mô hình 70 tỷ tham số, FP16 KV")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

Với cấu hình giả định, phép chia cho đúng con số của hình 7, ở đơn vị GB. Mỗi đường là một kích thước
batch, và nó cắt đường 45 GB ở độ dài ngữ cảnh tỉ lệ nghịch với batch: gấp đôi batch thì ngữ cảnh tối
đa còn một nửa. Đó là sự đánh đổi mà chương nêu: ngữ cảnh dài hơn đòi batch nhỏ hơn hoặc nhiều bộ nhớ
hơn, và thông lượng giảm theo.

## 5. Decode theo batch: bộ nhớ hay TPOT chặn trước?

Chương nói phép tính giá thành của nó chưa mô hình hoá năng lực của decode theo batch. Ô dưới dựng một
mô hình đơn giản, mọi phần đều là **giả định của sổ tay**: mỗi bước decode đọc 4 GB trọng số một lần
cho cả batch, cộng bộ nhớ đệm KV của từng chuỗi (trung bình 1,000 + 128 token, tức giữa lượt sinh),
với băng thông 3.35 TB/s; phần tính dùng tốc độ hiệu dụng suy ra ở phần 2; bước chậm hơn trong hai phần
quyết định, cộng thuế điều phối mỗi bước. Giá thuê GPU là 3 đô la mỗi giờ, theo chương.

**Dự đoán:** khi batch tăng, cái gì chặn trước: sức chứa 469 yêu cầu của bộ nhớ, hay đích TPOT 20 ms?

In [ ]:
THUE_BUOC = TPOT_MH - T_SAN
NGU_CANH_TB = PROMPT + TRA_LOI / 2
GIA_GIO = sach(3, nguon="469 yêu cầu · 3 đô la")
B_BO_NHO = math.floor(token_GiB / MOI_YEU_CAU)


def buoc_decode(b):
    doc = W_BYTE + b * NGU_CANH_TB * KV8
    return max(doc / BW_H100, FLOP_THAM_SO * THAM_SO * b / F_HD) + THUE_BUOC


cac_b = np.arange(1, 601)
tpot = np.array([buoc_decode(b) for b in cac_b])
token_s = cac_b / tpot
B_SLO = int(cac_b[tpot <= SLO_TPOT / 1000].max())
print(f"batch 1: TPOT {1000 * buoc_decode(1):.2f} ms, tức 1.57 ms của sách cộng phần đọc bộ nhớ đệm KV của một chuỗi")
print(f"đích TPOT 20 ms cho phép tối đa batch {B_SLO}; bộ nhớ cho phép {B_BO_NHO}")
for b in (1, 32, 128, B_SLO):
    doc_kv = b * NGU_CANH_TB * KV8
    print(f"batch {b:>3}: TPOT {1000 * buoc_decode(b):5.2f} ms · {b / buoc_decode(b):8,.0f} token/s"
          f" · bộ nhớ đệm KV chiếm {100 * doc_kv / (W_BYTE + doc_kv):3.0f} % lượng đọc mỗi bước"
          f" · {GIA_GIO / (b / buoc_decode(b) * 3600) * 1e6:.3f} đô la mỗi triệu token đầu ra")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.2))
ax1.plot(cac_b, 1000 * tpot, color=MAU[0], lw=2, label="mô hình: TPOT theo batch")
ax1.axhline(SLO_TPOT, color="0.3", lw=1, ls="--")
ax1.text(10, SLO_TPOT + 0.6, "đích TPOT 20 ms", fontsize=8, color="0.3")
ax1.axvline(B_BO_NHO, color=MAU[1], lw=1.2, ls=":")
ax1.text(B_BO_NHO + 6, 4, f"sức chứa bộ nhớ\n{B_BO_NHO} yêu cầu", fontsize=8, color=MAU[1])
ax1.set_xlabel("số chuỗi trong batch decode")
ax1.set_ylabel("TPOT (ms)")
ax1.set_ylim(0, 30)
ax1.set_title("Mỗi bước decode chậm dần khi batch lớn")
ax1.legend(loc="upper left", fontsize=8)
ax2.semilogy(cac_b, GIA_GIO / (token_s * 3600) * 1e6, color=MAU[2], lw=2)
ax2.axvline(B_SLO, color="0.3", lw=1, ls="--")
ax2.text(B_SLO - 8, 1.0, f"batch {B_SLO}:\nchạm đích TPOT", ha="right", fontsize=8, color="0.3")
ax2.set_xlabel("số chuỗi trong batch decode")
ax2.set_ylabel("đô la mỗi triệu token đầu ra (thang log)")
ax2.set_title("Giá mỗi token giảm theo batch")
plt.suptitle("Mô hình decode của sổ tay cho Llama 3 8B trên H100 (giả định)", fontsize=11)
plt.tight_layout()
plt.show()

Trong mô hình này, decode nằm trên dốc băng thông ở mọi batch, nhưng thứ chiếm băng thông đổi dần: ở
batch nhỏ là trọng số, ở batch lớn là bộ nhớ đệm KV của từng chuỗi. Mỗi chuỗi thêm vào kéo TPOT lên một
chút, nên đích TPOT 20 ms chặn batch **trước** khi bộ nhớ đầy. Cùng lúc, giá mỗi token giảm hơn ba mươi
lần từ batch 1 tới đó, phần lớn ngay trong vài chục chuỗi đầu, vì 4 GB trọng số được chia cho nhiều chuỗi
hơn. Đây là lý do chương nói phục vụ LLM là
quản lý bộ nhớ: bố cục bộ nhớ đệm KV và cách gom batch quyết định cả số người dùng cùng lúc lẫn giá mỗi
token. Mô hình bỏ qua tranh chấp giữa prefill và decode trên cùng GPU, đúng chỗ mà chương cũng nói phép
tính của nó chưa mô hình hoá.

## 6. Gom batch liên tục

Gom batch động ở sổ tay trước giả định đầu ra có độ dài cố định. Chương đưa một ví dụ: trong một batch
tám chuỗi, một chuỗi xong sau mười token còn các chuỗi khác cần 100 token, nên khe của chuỗi ấy phí phần
lớn tính toán. Gom batch liên tục cho yêu cầu mới vào khe trống ngay ở bước kế tiếp.

In [ ]:
CAN = sach(100, trich="finishes after ten tokens while others need 100 tokens")
XONG_SOM = 10                      # "ten tokens": sách viết bằng chữ, nên không trích được như một con số
theo_sach("phần tính toán phí của khe ấy (%)", 100 * (CAN - XONG_SOM) / CAN, sach=90, nguon="90 %")

Một ví dụ chưa nói được batch tĩnh phí bao nhiêu trên một tải công việc thật. Ô dưới mô phỏng hai bộ lập
lịch với 8 khe decode. Độ dài câu trả lời là **giả định của sổ tay**: phân phối log-chuẩn với trung vị
120 token, cắt trong khoảng 1 tới 1,000 token, để có vài câu trả lời rất dài giữa nhiều câu ngắn, như
chương mô tả (từ một token tới hàng nghìn token). Thời gian đo bằng bước decode.

* **Gom batch tĩnh:** một batch nhận tới 8 yêu cầu đang chờ, và chạy cho tới khi chuỗi dài nhất xong.
* **Gom batch liên tục:** khe nào xong thì nhận ngay yêu cầu kế tiếp.

**Dự đoán:** với 10,000 yêu cầu chờ sẵn, gom batch tĩnh dùng được bao nhiêu phần trăm số khe-bước?

In [ ]:
KHE = 8                            # tám khe, như ví dụ của sách
rng = np.random.default_rng(42)
DAI = np.clip(np.round(rng.lognormal(math.log(120), 1.0, 10_000)), 1, 1000).astype(int)


def tinh_tai(den, dai, khe=KHE):
    """Gom batch tĩnh: khi batch trước xong, nhận tới `khe` yêu cầu đã tới; chạy hết chuỗi dài nhất."""
    n = len(den)
    cho, ranh, i, ban = np.empty(n), 0.0, 0, 0.0
    lich = []
    while i < n:
        bd = max(ranh, den[i])
        j = max(i + 1, min(i + khe, int(np.searchsorted(den, bd, side="right"))))
        cho[i:j] = bd - den[i:j]
        lich.append((bd, i, j))
        ranh = bd + dai[i:j].max()
        ban += khe * dai[i:j].max()
        i = j
    return cho, dai.sum() / ban, lich


def lien_tuc(den, dai, khe=KHE):
    """Gom batch liên tục: mỗi yêu cầu vào khe rảnh sớm nhất, theo thứ tự đến."""
    ranh = [(0.0, s) for s in range(khe)]
    cho, lich = np.empty(len(den)), []
    for i, (t, d) in enumerate(zip(den, dai)):
        r, s = heapq.heappop(ranh)
        bd = max(r, t)
        cho[i] = bd - t
        lich.append((s, bd, d))
        heapq.heappush(ranh, (bd + d, s))
    xong = max(r for r, _ in ranh)
    return cho, dai.sum() / (khe * xong), lich


cho_san = np.zeros(len(DAI))                          # tất cả chờ sẵn ở bước 0
_, dung_tinh, _ = tinh_tai(cho_san, DAI)
_, dung_lt, _ = lien_tuc(cho_san, DAI)
print(f"độ dài câu trả lời: trung bình {DAI.mean():.0f}, trung vị {np.median(DAI):.0f}, dài nhất {DAI.max()} token")
print(f"10,000 yêu cầu chờ sẵn: gom batch tĩnh dùng {dung_tinh:.1%} số khe-bước, gom batch liên tục {dung_lt:.1%}")
print(f"cùng một GPU, gom batch liên tục xong việc nhanh hơn {dung_lt / dung_tinh:.1f} lần")

NANG_LUC = KHE / DAI.mean()                           # yêu cầu mỗi bước khi mọi khe luôn bận
for phan in (0.2, 0.3, 0.6, 0.9):
    den = np.cumsum(rng.exponential(1 / (phan * NANG_LUC), len(DAI)))
    cho_lt, _, _ = lien_tuc(den, DAI)
    dong = f"tải {phan:.0%} năng lực: liên tục chờ trung bình {cho_lt.mean():7.1f} bước, p99 {np.percentile(cho_lt, 99):7.1f}"
    if phan < dung_tinh:
        cho_t, _, _ = tinh_tai(den, DAI)
        dong += f" · tĩnh chờ trung bình {cho_t.mean():7.1f}, p99 {np.percentile(cho_t, 99):7.1f}"
    else:
        dong += f" · tĩnh: quá năng lực của nó (khoảng {dung_tinh:.0%}), hàng đợi dài ra mãi"
    print(dong)

In [ ]:
mau_nho = DAI[:20]
_, _, lich_t = tinh_tai(np.zeros(20), mau_nho)
_, _, lich_l = lien_tuc(np.zeros(20), mau_nho)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 4.8), sharex=True)
for bd, i, j in lich_t:
    dai_nhat = mau_nho[i:j].max()
    for s, k in enumerate(range(i, j)):
        ax1.broken_barh([(bd, mau_nho[k])], (s - 0.4, 0.8), facecolors=MAU[0], edgecolor="white", lw=1)
        if mau_nho[k] < dai_nhat:
            ax1.broken_barh([(bd + mau_nho[k], dai_nhat - mau_nho[k])], (s - 0.4, 0.8), facecolors="0.88", lw=0)
for s, bd, d in lich_l:
    ax2.broken_barh([(bd, d)], (s - 0.4, 0.8), facecolors=MAU[0], edgecolor="white", lw=1)
for ax, ten in ((ax1, "Gom batch tĩnh: khe chờ chuỗi dài nhất (xám)"), (ax2, "Gom batch liên tục: khe xong là nhận việc mới")):
    ax.set_yticks(range(KHE), [f"khe {s + 1}" for s in range(KHE)], fontsize=8)
    ax.set_title(ten, fontsize=10)
    ax.set_ylim(-0.6, KHE - 0.4)
    ax.grid(False)
ax2.set_xlabel("bước decode (20 yêu cầu đầu tiên, chờ sẵn)")
plt.tight_layout()
plt.show()

Với phân phối độ dài này, gom batch tĩnh chỉ dùng chừng một phần ba số khe-bước: mỗi batch chạy theo
chuỗi dài nhất, và vì độ dài trải rộng, chuỗi dài nhất trong tám chuỗi thường dài gấp mấy lần trung bình.
Phần còn lại là khe trống (vùng xám). Hệ quả với lưu lượng thật còn nặng hơn: ở mức tải mà gom batch
liên tục còn dư sức, gom batch tĩnh đã quá năng lực và hàng đợi dài ra mãi. Lợi ích đến từ đúng hai nguồn
chương nêu: bớt tính toán phí cho chuỗi đã xong, và bớt thời gian chờ của yêu cầu mới.

### PagedAttention: cấp bộ nhớ đệm KV theo khối

Gom batch liên tục làm việc cấp phát khó thêm, vì bộ nhớ đệm KV phải được cấp và trả liên tục. Một cách
cài đặt đơn giản giữ trước cho mỗi chuỗi một vùng liền đủ cho độ dài tối đa, vì không biết trước câu trả
lời dài bao nhiêu. Chương nói cách ấy có thể phí 40 tới 50 % bộ nhớ, còn PagedAttention, chia bộ nhớ
đệm thành các khối cố định ánh xạ qua một bảng khối, giữ mức phí dưới 4 % trong bài báo vLLM.

Ô dưới so hai cách trên cùng một tải giả định của sổ tay: prompt log-chuẩn với trung vị 300 token, câu
trả lời như ở trên, độ dài tối đa 2,048 token, khối 16 token. Phí được đo ở một thời điểm ngẫu nhiên giữa
lượt sinh, khi câu trả lời mới sinh được một phần.

In [ ]:
rng = np.random.default_rng(7)
prompt_tai = np.clip(np.round(rng.lognormal(math.log(300), 0.8, len(DAI))), 16, 1500).astype(int)
TOI_DA, KHOI = 2048, 16                              # giả định của sổ tay
dang_dung = np.minimum(prompt_tai + np.floor(rng.uniform(0, 1, len(DAI)) * DAI).astype(int) + 1, TOI_DA)
lien_khoi = np.full(len(DAI), TOI_DA)                 # giữ trước độ dài tối đa
theo_khoi = np.ceil(dang_dung / KHOI) * KHOI          # chỉ phần khối cuối còn trống
print(f"giữ trước vùng liền {TOI_DA} token: phí {1 - dang_dung.sum() / lien_khoi.sum():.1%} bộ nhớ đệm đã cấp")
print(f"cấp theo khối {KHOI} token: phí {1 - dang_dung.sum() / theo_khoi.sum():.1%}")
print(f"sách: {sach(40, nguon='150 token · 40 · 50 %')} tới {sach(50, nguon='40 · 50 %')} % với cách cài đặt đơn giản,"
      f" dưới {sach(4, nguon='50 % · 4 %')} % với PagedAttention")
print(f"trong {token_GiB:,.0f} token của phần dự trữ: giữ trước vùng liền chứa {int(token_GiB // TOI_DA)} chuỗi,"
      f" cấp theo khối chứa khoảng {token_GiB / theo_khoi.mean():,.0f} chuỗi")

Mức phí của cách giữ trước tuỳ hoàn toàn vào phân phối độ dài so với độ dài tối đa; với tải giả định này
nó còn cao hơn khoảng 40 tới 50 % mà chương nêu, vì phần lớn chuỗi ngắn hơn nhiều so với 2,048 token. Cấp
theo khối thì chỉ phí phần trống của khối cuối, trung bình nửa khối mỗi chuỗi, nên mức phí nhỏ bất kể
phân phối. Bộ nhớ không còn bị giữ chỗ, nên cùng phần dự trữ chứa được nhiều chuỗi hơn hẳn, và batch
decode lớn hơn đồng nghĩa với giá mỗi token thấp hơn, như phần 5.

## 7. Lưu đệm tiền tố và chuyển bộ nhớ đệm ra ngoài

Lưu đệm tiền tố giữ trạng thái KV của những tiền tố chung, như một prompt hệ thống 2,000 token. Với $N$
yêu cầu chung một tiền tố dài $S$ token, phần prefill tránh được xấp xỉ $(N-1)\,S$ bước token. Ô dưới
lấy $N = 100$ yêu cầu với 200 token riêng mỗi yêu cầu (giả định của sổ tay), và đổi số token ra thời gian
bằng tốc độ hiệu dụng suy ra ở phần 2, bỏ qua phần tính tập trung tăng theo độ dài.

In [ ]:
S_TIEN_TO = sach(2000, nguon="2,000 token")
N_YC, RIENG = 100, 200                                # giả định của sổ tay
tranh = (N_YC - 1) * S_TIEN_TO
tong_prefill = N_YC * (S_TIEN_TO + RIENG)
print(f"tránh được {tranh:,} bước token, {tranh / tong_prefill:.0%} tổng phần prefill của {N_YC} yêu cầu")


def giay(so_token):
    return FLOP_THAM_SO * THAM_SO * so_token / F_HD


print(f"TTFT phía mô hình mỗi yêu cầu: {1000 * giay(S_TIEN_TO + RIENG):.0f} ms không lưu đệm,"
      f" {1000 * giay(RIENG):.1f} ms khi tiền tố đã có trong bộ nhớ đệm")

PCIE = sach(32, nguon="10 GB · 32 GB/s") * GB
for so_token in (MOI_YEU_CAU, S_TIEN_TO, 32_000):
    print(f"nạp lại {so_token:>6,} token bộ nhớ đệm KV ({so_token * KV8 / 1e6:7.1f} MB) qua PCIe 32 GB/s:"
          f" ít nhất {1000 * so_token * KV8 / PCIE:6.1f} ms")

Khi bộ nhớ đệm KV vượt VRAM, hệ thống có thể chuyển những ngữ cảnh không hoạt động sang RAM của máy chủ.
Chi phí nạp lại tối thiểu là số byte chia cho băng thông của đường truyền, trước khi cộng phụ trội phần
mềm và hàng đợi. Nạp lại một hội thoại ngắn chỉ mất vài mili giây, rẻ hơn tính lại phần prefill; một
ngữ cảnh rất dài thì tốn hơn một trăm mili giây, quá nửa ngân sách TTFT 200 ms. Đó là lý do chương dặn tính rõ rủi
ro độ trễ ấy vào ngân sách.

## 8. Năng lượng của một cuộc trò chuyện

Napkin math 1.5 giả định H100 dùng đủ công suất thiết kế nhiệt 700 W, cộng 300 W cho máy chủ, với 114
yêu cầu đồng thời, mỗi yêu cầu 8 token mỗi giây.

In [ ]:
P_GPU = sach(700, nguon="1.5 · 700 W")
P_MAY = sach(300, nguon="700 W · 300 W")
theo_sach("công suất tổng (W)", P_GPU + P_MAY, sach=1000, nguon="300 W · 1,000 W")
DONG_THOI = sach(114, nguon="1,000 W · 114 yêu cầu")
NHIP = sach(8, nguon="114 yêu cầu · 8 token")
THONG_LUONG = DONG_THOI * NHIP
theo_sach("thông lượng (token/s)", THONG_LUONG, sach=912, nguon="8 token · 912 token")
J_TOKEN = (P_GPU + P_MAY) / THONG_LUONG
theo_sach("năng lượng mỗi token (J)", J_TOKEN, sach=1.0965, nguon="912 token · 1.0965 jun")
DAI_TL = sach(500, nguon="1.0965 jun · 500 token")
theo_sach("một câu trả lời 500 token (J)", DAI_TL * J_TOKEN, sach=548.2, nguon="500 token · 548.2 jun")
SAC = sach(40_000, nguon="548.2 jun · 40,000 jun")
NUOC = sach(100_000, nguon="40,000 jun · 100,000 jun")
print(f"một lần sạc điện thoại ≈ {SAC / (DAI_TL * J_TOKEN):.0f} câu trả lời; đun sôi một cốc nước ≈ {NUOC / (DAI_TL * J_TOKEN):.0f} câu trả lời")
MUC_THAP = sach(10, nguon="100,000 jun · 10 %") / 100
P_GPU_THAP = sach(300, nguon="700 W · 300 W")       # GPU vẫn tiêu 300 W khi gom batch kém
J_THAP = (P_GPU_THAP + P_MAY) / (THONG_LUONG * MUC_THAP)
theo_sach("năng lượng mỗi token ở 10 % mức sử dụng (J)", J_THAP, sach=6.6, nguon="10 % · 6.6 jun")
theo_sach("so với mức cơ sở (lần)", J_THAP / J_TOKEN, sach=6, nguon="6.6 jun · 6 lần")

Sách cho hai điểm: đủ tải (1,000 W, 912 token mỗi giây) và 10 % mức sử dụng (600 W, một phần mười số
token). Ô dưới nối hai điểm ấy bằng một giả định của sổ tay: công suất GPU tuyến tính theo mức sử dụng,
đi qua 300 W ở 10 % và 700 W ở 100 %, còn thông lượng tỉ lệ với mức sử dụng.

**Dự đoán:** ở 50 % mức sử dụng, năng lượng mỗi token gấp mấy lần mức cơ sở?

In [ ]:
muc = np.linspace(0.05, 1.0, 200)
doc_cong_suat = (P_GPU - P_GPU_THAP) / (1 - MUC_THAP)
p_gpu = P_GPU_THAP + doc_cong_suat * (muc - MUC_THAP)
j_muc = (p_gpu + P_MAY) / (THONG_LUONG * muc)
o50 = int(np.argmin(np.abs(muc - 0.5)))
print(f"công suất GPU giả định ở 0 %: {P_GPU_THAP - doc_cong_suat * MUC_THAP:.0f} W")
print(f"ở 50 %: {j_muc[o50]:.2f} J mỗi token, gấp {j_muc[o50] / J_TOKEN:.2f} lần mức cơ sở")
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot(100 * muc, j_muc, color=MAU[0], lw=2, label="công suất tuyến tính theo mức sử dụng (giả định)")
ax.plot([100, 100 * MUC_THAP], [J_TOKEN, J_THAP], "o", color=MAU[1], ms=8, mec="white", mew=1.5,
        label="hai điểm của napkin math 1.5")
ax.annotate("1.0965 J: đủ tải", (100, J_TOKEN), xytext=(-60, 30), textcoords="offset points", fontsize=8,
            arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.annotate("6.6 J: 10 % mức sử dụng", (100 * MUC_THAP, J_THAP), xytext=(25, 10), textcoords="offset points", fontsize=8,
            arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.set_ylim(0, 14)
ax.set_xlim(0, 103)
ax.set_xlabel("mức sử dụng GPU (%)")
ax.set_ylabel("năng lượng mỗi token (J)")
ax.set_title("Năng lượng mỗi token theo mức sử dụng")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

Năng lượng mỗi token giảm nhanh khi mức sử dụng tăng, vì phần công suất nền (máy chủ, GPU lúc nhàn rỗi)
được chia cho nhiều token hơn. Cách chính để giảm jun mỗi token là tăng mức sử dụng: mỗi bước decode đọc
trọng số một lần cho cả batch đang chạy, nên gom batch tốt là chỗ tiết kiệm năng lượng lớn nhất. Khác
với suy luận có đầu ra cố định, năng lượng của một câu trả lời tăng theo độ dài của nó.

## 9. Giá mỗi triệu token

Chương lấy giá thuê H100 SXM5 khoảng 3 đô la mỗi giờ và một trần thông lượng lạc quan 45.2 triệu token
mỗi giờ, là số nhỏ hơn giữa năng lực prefill riêng và năng lực giữ bộ nhớ đệm KV.

In [ ]:
TRAN_GIO = sach(45.2e6, nguon="3 đô la · 45.2 triệu")
theo_sach("giá mỗi triệu token (đô la)", GIA_GIO / (TRAN_GIO / 1e6), sach=0.066, nguon="45.2 triệu · 0.066 đô la")
print(f"45.2 triệu token mỗi giờ = {TRAN_GIO / 3600:,.0f} token/s = {TRAN_GIO / 3600 / MOI_YEU_CAU:.1f} yêu cầu"
      f" {MOI_YEU_CAU:,} token mỗi giây")
print(f"so sánh, prefill riêng ở 41.9 ms mỗi yêu cầu: {1 / TTFT_MH:.1f} yêu cầu mỗi giây nếu GPU không làm gì khác")

Chương không nêu các bước dẫn tới 45.2 triệu, nên ô trên chỉ đổi nó ra đơn vị dễ hình dung: khoảng mười
yêu cầu mỗi giây cho một GPU. Con số ấy thấp hơn năng lực prefill riêng mà 41.9 ms ngầm định, và thấp hơn
nhiều so với batch decode của phần 5, nên bộ giải của sách đã dùng những giả định mà chương không nêu. Sách cũng nói phép tính chưa mô hình hoá tranh chấp giữa prefill và decode. Bài học vẫn đứng: dung
lượng bộ nhớ đệm KV quyết định số yêu cầu nằm cùng lúc, còn tính toán của prefill và băng thông của decode
quyết định thông lượng và giá thật.

## Thử thêm

1. Ở ô `decode_model`, đổi `NGU_CANH_TB` thành `8_000 + 128` (prompt dài, như một ngữ cảnh RAG). Batch
   lớn nhất còn đạt đích TPOT là bao nhiêu, và giá mỗi triệu token tăng bao nhiêu lần?
2. Ở ô `kv`, thử `kv_moi_token(LLAMA3_8B, byte=1)` (bộ nhớ đệm KV 8 bit). Sức chứa đổi ra sao? Chương nói
   AWQ chỉ lượng tử hóa trọng số, nên đây là một kỹ thuật khác, không phải thứ AWQ cho không.
3. Ở ô `cb_sim`, đổi `sigma` của phân phối log-chuẩn từ `1.0` thành `0.2` (độ dài gần như đều nhau). Gom
   batch tĩnh còn phí bao nhiêu? Điều đó nói gì về những tải công việc mà gom batch liên tục đáng công?
4. Ở ô `paged`, đổi `KHOI` thành 128 rồi 1. Khối lớn phí ở đâu, khối nhỏ tốn ở đâu (gợi ý: bảng khối)?

## Tóm lại

* Một yêu cầu LLM có hai pha: prefill quyết định TTFT và chạy như một batch lớn, decode quyết định TPOT
  và đọc toàn bộ trọng số ở mỗi bước. Với Llama 3 8B INT4 trên H100, mức sàn là 4 GB chia 3.35 TB/s,
  khoảng 1.2 ms mỗi token.
* Decode ở batch nhỏ nằm trên dốc băng thông, nên gom batch tăng thông lượng gần như miễn phí; phép đo
  trên CPU cho cùng hình dạng, chỉ với điểm gãy gần hơn nhiều.
* Bộ nhớ đệm KV tỉ lệ với batch nhân độ dài ngữ cảnh. Số 0.6 triệu token và 469 yêu cầu của sách chỉ khép
  lại khi đọc 72 GB là GiB.
* Trong mô hình decode của sổ tay, đích TPOT chặn batch trước khi bộ nhớ đầy, còn giá mỗi token giảm hơn
  ba mươi lần từ batch 1 tới đó.
* Gom batch tĩnh chạy theo chuỗi dài nhất và phí phần lớn khe khi độ dài trải rộng; gom batch liên tục và
  cấp bộ nhớ đệm theo khối lấy lại phần phí ấy.
* Năng lượng và giá mỗi token cùng giảm khi mức sử dụng tăng: 1.0965 J ở đủ tải, 6.6 J ở 10 %.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Phục vụ mô hình — mỗi yêu cầu một ngân sách độ trễ](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch13-model-serving/), dựng từ chương
[*Model Serving*](https://mlsysbook.ai/vol1/model_serving/model_serving.html) của sách.

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.